# 🕵️ FraudSpotter — Fake Review Detection
### Part 1: Data Exploration & Preprocessing
---
> **Dataset:** 40,000 Amazon reviews labeled as `CG` (Computer-Generated/Fake) or `OR` (Original/Human)  
> **Goal:** Clean and preprocess text data for ML model training  
> **Tech Stack:** Python | Pandas | NLTK | Seaborn | Matplotlib


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import string, warnings, nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

warnings.filterwarnings('ignore')
%matplotlib inline

# Style settings — makes all plots look professional
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#c9d1d9',
    'ytick.color':      '#c9d1d9',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   14,
    'axes.titleweight': 'bold',
})

print("✅ All libraries loaded successfully!")


In [ ]:
# ── Download NLTK resources ───────────────────────────────────────────────────
for pkg in ['stopwords', 'punkt', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, quiet=True)
print("✅ NLTK resources ready!")


## 📂 Step 1: Load Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('fake reviews dataset.csv')

print(f"📊 Dataset Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"🏷️  Label Classes   : {df['label'].unique().tolist()}")
print(f"📝 Columns         : {df.columns.tolist()}")
print()
df.head(5)


## 🔍 Step 2: Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print("=" * 50)
print("  DATASET OVERVIEW")
print("=" * 50)
print(f"  Total Reviews  : {len(df):,}")
print(f"  Missing Values : {df.isnull().sum().sum()}")
print(f"  Data Types     :\n{df.dtypes}")
print("=" * 50)


In [ ]:
# ── Label Distribution ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Fake Review Dataset — Label Distribution', 
             fontsize=16, fontweight='bold', color='#c9d1d9', y=1.02)

label_counts = df['label'].value_counts()
colors = ['#f85149', '#3fb950']  # red = fake (CG), green = real (OR)
explode = (0.05, 0.05)

# Pie chart
axes[0].pie(label_counts.values, labels=['Computer-Generated (Fake)', 'Original (Real)'],
            colors=colors, explode=explode, autopct='%1.1f%%',
            startangle=90, textprops={'color': '#c9d1d9', 'fontsize': 11},
            wedgeprops={'edgecolor': '#0d1117', 'linewidth': 2})
axes[0].set_title('Label Split', color='#c9d1d9', fontsize=13, pad=12)
axes[0].set_facecolor('#161b22')

# Bar chart
bars = axes[1].bar(label_counts.index, label_counts.values, color=colors, 
                    edgecolor='#0d1117', linewidth=1.5, width=0.5)
for bar, val in zip(bars, label_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', va='bottom', color='#c9d1d9', fontsize=12, fontweight='bold')
axes[1].set_title('Review Count by Label', color='#c9d1d9', fontsize=13, pad=12)
axes[1].set_xlabel('Label', color='#c9d1d9')
axes[1].set_ylabel('Count', color='#c9d1d9')
axes[1].set_ylim(0, label_counts.max() * 1.15)

plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"\n📊 CG (Fake)   : {label_counts.get('CG', 0):,} reviews")
print(f"📊 OR (Real)   : {label_counts.get('OR', 0):,} reviews")


In [ ]:
# ── Rating Distribution ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Rating Distribution', fontsize=16, fontweight='bold', color='#c9d1d9')

rating_colors = ['#f85149','#e3b341','#e3b341','#3fb950','#3fb950']
rating_counts = df['rating'].value_counts().sort_index()

# Overall ratings
axes[0].bar(rating_counts.index, rating_counts.values, color=rating_colors,
            edgecolor='#0d1117', linewidth=1.5, width=0.6)
axes[0].set_title('Overall Rating Distribution', color='#c9d1d9', fontsize=12)
axes[0].set_xlabel('Star Rating ⭐', color='#c9d1d9')
axes[0].set_ylabel('Count', color='#c9d1d9')

# Ratings by label
for label, color in zip(['CG', 'OR'], ['#f85149', '#3fb950']):
    subset = df[df['label'] == label]['rating'].value_counts().sort_index()
    axes[1].plot(subset.index, subset.values, marker='o', linewidth=2.5,
                 markersize=8, color=color, label=f'{"Fake (CG)" if label=="CG" else "Real (OR)"}')

axes[1].set_title('Ratings by Label', color='#c9d1d9', fontsize=12)
axes[1].set_xlabel('Star Rating ⭐', color='#c9d1d9')
axes[1].set_ylabel('Count', color='#c9d1d9')
axes[1].legend(facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9')

plt.tight_layout()
plt.savefig('rating_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()


## 🧹 Step 3: Text Cleaning & Preprocessing

In [ ]:
# ── Text Cleaning Pipeline ────────────────────────────────────────────────────
stop_words = set(stopwords.words('english'))
stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    """Remove punctuation from text."""
    return ''.join([ch for ch in str(text) if ch not in string.punctuation])

def preprocess(text):
    """Tokenize → remove stopwords & digits → clean tokens."""
    tokens = word_tokenize(str(text))
    return ' '.join([w for w in tokens 
                     if w.lower() not in stop_words 
                     and not w.isdigit() 
                     and w.isalpha()])

def stem_words(text):
    """Apply Porter Stemming to reduce words to root form."""
    return ' '.join([stemmer.stem(w) for w in str(text).split()])

def lemmatize_words(text):
    """Apply Lemmatization for better root word extraction."""
    return ' '.join([lemmatizer.lemmatize(w) for w in str(text).split()])

# Demo on one review
sample = df['text_'][0]
print("📝 Original text  :", sample[:150])
print()
print("🧹 After cleaning :", clean_text(sample)[:150])
print()
print("✂️  After preprocessing:", preprocess(sample)[:150])


In [ ]:
# ── Apply Full Preprocessing Pipeline ────────────────────────────────────────
print("⏳ Applying preprocessing pipeline...")

df['text_'] = df['text_'].astype(str)

# Process in chunks to avoid memory issues
chunk = 10000
for start in range(0, len(df), chunk):
    end = min(start + chunk, len(df))
    df.loc[start:end-1, 'text_'] = df.loc[start:end-1, 'text_'].apply(preprocess)
    print(f"  ✅ Processed rows {start:,} → {end:,}")

# Lowercase → Stem → Lemmatize
df['text_'] = df['text_'].str.lower()
df['text_'] = df['text_'].apply(stem_words)
df['text_'] = df['text_'].apply(lemmatize_words)

print()
print("🎉 Preprocessing complete!")
print(f"   Sample processed text: {df['text_'][0][:120]}")


## 💾 Step 4: Save Preprocessed Data

In [ ]:
# Save the cleaned dataset
df.to_csv('Preprocessed Fake Reviews Detection Dataset.csv', index=False)

print("✅ Preprocessed dataset saved!")
print(f"   Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print()
print("📋 Preview of cleaned data:")
df[['text_', 'label', 'rating']].head(5)
